# Pocket FM — Play Store Review Mining & Growth Insight Project

**Goal:** Fetch real user reviews for Pocket FM from the Google Play Store, clean them, and mine them for growth-relevant insights (churn signals, complaint themes, sentiment trends) — the same kind of analysis a Growth Analyst at Pocket FM would run internally.

**Pipeline:** Fetch → Clean → Feature Engineer → Analyze → Sentiment → Theme Extraction → Growth Insight Summary

**App analyzed:** Pocket FM: Audio Series — Play Store package ID: `com.radio.pocketfm`

---
### ⚠️ Before you start
Run every cell **top to bottom, in order** — later cells depend on variables created earlier. If a cell errors, re-read the error message; it usually tells you exactly what's missing (e.g., a library not installed).

## Step 0 — One-time setup (do this only the first time)

You said Jupyter and the libraries aren't installed yet, so let's do that first — **from your computer's terminal / command prompt** (not inside a notebook), run these one at a time:

```bash
# 1. Check Python is installed (should show 3.9 or higher)
python3 --version

# 2. Install Jupyter Notebook
pip install notebook

# 3. Install all the libraries this project needs
pip install google-play-scraper pandas matplotlib seaborn wordcloud vaderSentiment langdetect emoji

# 4. Launch Jupyter (run this from the folder where you want to save this project)
jupyter notebook
```

This will open Jupyter in your browser. Create a new notebook there (or open this `.ipynb` file directly if I've sent you one), and continue below.

**If `pip` doesn't work**, try `pip3` instead. **If `python3` doesn't work**, you likely don't have Python installed — download it from python.org first (tick "Add Python to PATH" during install on Windows).

The cell below double-checks the installs worked — run it first inside Jupyter.

In [ ]:
# Run this once to confirm everything is installed correctly.
# If any import fails, go back to Step 0 and pip install the missing library.

import importlib
libs = ["google_play_scraper", "pandas", "matplotlib", "seaborn",
        "wordcloud", "vaderSentiment", "langdetect", "emoji"]

for lib in libs:
    try:
        importlib.import_module(lib)
        print(f"OK   - {lib}")
    except ImportError:
        print(f"MISSING - {lib}  --> run: pip install {lib}")

## Step 1 — Imports

Now we import everything we'll actually use in the analysis.

In [ ]:
import pandas as pd
import numpy as np
import re
import emoji
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter

from google_play_scraper import Sort, reviews, app as gp_app
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from langdetect import detect, DetectorFactory
from wordcloud import WordCloud

DetectorFactory.seed = 0  # makes langdetect results reproducible

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

APP_ID = "com.radio.pocketfm"   # Pocket FM: Audio Series
print("Setup complete.")

## Step 2 — Pull basic app info (sanity check)

Before scraping thousands of reviews, let's confirm we have the right app and see its current overall rating, install count, etc. This also doubles as your "context slide" if you ever present this project.

In [ ]:
info = gp_app(APP_ID, lang="en", country="in")

print("App name       :", info["title"])
print("Overall rating :", info["score"])
print("Total ratings  :", info["ratings"])
print("Installs       :", info["installs"])
print("Genre          :", info["genre"])

## Step 3 — Fetch reviews (the actual "data fetching" step)

`google-play-scraper` paginates results using a `continuation_token`. We loop and keep calling it until we've collected enough reviews or run out.

**Note:** Play Store only serves the most recent reviews (typically several thousand) — that's expected and totally fine for this project, it's not a data-quality error on your part.

This will take 1-3 minutes depending on your internet connection.

In [ ]:
def fetch_reviews(app_id, total_target=3000, lang="en", country="in", sort=Sort.NEWEST):
    all_reviews = []
    token = None
    batch_size = 200

    while len(all_reviews) < total_target:
        result, token = reviews(
            app_id,
            lang=lang,
            country=country,
            sort=sort,
            count=batch_size,
            continuation_token=token
        )
        if not result:
            break  # no more reviews available
        all_reviews.extend(result)
        print(f"Fetched so far: {len(all_reviews)}")
        if token is None:
            break  # reached the end

    return all_reviews[:total_target]

raw_reviews = fetch_reviews(APP_ID, total_target=3000)
print(f"\nTotal reviews fetched: {len(raw_reviews)}")

**Getting more data (optional):** Play Store reviews are region- and language-specific. To get a richer, more realistic dataset, fetch a second batch in Hindi and combine it — this also gives you a genuine multilingual-cleaning story for your interview.

In [ ]:
raw_reviews_hi = fetch_reviews(APP_ID, total_target=1000, lang="hi", country="in")
print(f"Additional Hindi reviews fetched: {len(raw_reviews_hi)}")

all_raw = raw_reviews + raw_reviews_hi
print(f"Combined total: {len(all_raw)}")

## Step 4 — Convert to DataFrame and save the raw data

Always save your raw, untouched data to disk before cleaning it. If you make a cleaning mistake later, you can always restart from this file instead of re-scraping.

In [ ]:
df_raw = pd.DataFrame(all_raw)
print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.to_csv("pocketfm_reviews_raw.csv", index=False)
print("Saved raw data to pocketfm_reviews_raw.csv")

## Step 5 — Initial exploration (before cleaning)

Always look at your raw data before touching it. We check: shape, column meanings, missing values, and duplicates.

In [ ]:
df_raw.info()

In [ ]:
# Key columns we actually care about, with clearer names
df = df_raw[[
    "reviewId", "userName", "content", "score",
    "thumbsUpCount", "at", "replyContent", "repliedAt"
]].copy()

df.rename(columns={
    "content": "review_text",
    "score": "rating",
    "at": "review_date"
}, inplace=True)

df.head()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate reviewIds:", df['reviewId'].duplicated().sum())
print("Duplicate review texts:", df['review_text'].duplicated().sum())

## Step 6 — Data Cleaning

This is the core "data cleaning" part of the project. We handle, in order:

1. Drop exact duplicate reviews (same reviewId or identical text)
2. Drop rows with missing/empty review text (can't analyze what isn't there)
3. Standardize the date column
4. Remove emojis into a separate flag (useful signal, but noisy for text analysis)
5. Clean text: lowercase, strip URLs/special characters, remove extra whitespace
6. Detect language per review (Pocket FM has a large Hindi + regional user base — expect a real mix)

In [ ]:
# 1. Remove duplicates
before = len(df)
df.drop_duplicates(subset="reviewId", inplace=True)
df.drop_duplicates(subset="review_text", inplace=True)
print(f"Removed {before - len(df)} duplicate reviews")

# 2. Drop empty/missing review text
before = len(df)
df = df[df["review_text"].notnull()]
df = df[df["review_text"].str.strip() != ""]
print(f"Removed {before - len(df)} empty reviews")

In [ ]:
# 3. Standardize date column
df["review_date"] = pd.to_datetime(df["review_date"])
df["year_month"] = df["review_date"].dt.to_period("M").astype(str)
print(df["review_date"].min(), "to", df["review_date"].max())

In [ ]:
# 4. Flag whether a review contains emojis, then strip them for text analysis
df["has_emoji"] = df["review_text"].apply(lambda x: bool(emoji.emoji_count(x)))
df["review_text_clean"] = df["review_text"].apply(lambda x: emoji.replace_emoji(x, replace=""))

print(f"Reviews containing emojis: {df['has_emoji'].sum()} ({df['has_emoji'].mean():.1%})")

In [ ]:
# 5. General text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", "", text)      # remove URLs
    text = re.sub(r"[^a-zA-Z0-9\u0900-\u097F\s]", " ", text)  # keep letters/numbers + Devanagari (Hindi) script
    text = re.sub(r"\s+", " ", text).strip()          # collapse whitespace
    return text

df["review_text_clean"] = df["review_text_clean"].apply(clean_text)
df = df[df["review_text_clean"].str.strip() != ""]  # drop rows that became empty after cleaning
df[["review_text", "review_text_clean"]].head()

In [ ]:
# 6. Language detection (multilingual handling — a genuine data-cleaning challenge)
def safe_detect(text):
    try:
        if len(text.split()) < 2:   # too short to detect reliably
            return "unknown"
        return detect(text)
    except Exception:
        return "unknown"

df["language"] = df["review_text_clean"].apply(safe_detect)
print(df["language"].value_counts().head(10))

**Design decision:** rather than translating Hindi/regional reviews (which risks losing meaning and adds complexity), we'll keep them tagged by language and analyze English reviews for keyword/sentiment depth, while still using language mix itself as an insight (e.g., "X% of reviews are in Hindi — localization of in-app support could matter").

## Step 7 — Feature Engineering

Adding a few derived columns that make the analysis richer.

In [ ]:
df["review_length"] = df["review_text_clean"].apply(lambda x: len(x.split()))

def rating_bucket(r):
    if r <= 2:
        return "Negative (1-2)"
    elif r == 3:
        return "Neutral (3)"
    else:
        return "Positive (4-5)"

df["rating_bucket"] = df["rating"].apply(rating_bucket)

df[["rating", "rating_bucket", "review_length", "language", "has_emoji"]].head()

In [ ]:
df.to_csv("pocketfm_reviews_cleaned.csv", index=False)
print(f"Final cleaned dataset: {df.shape[0]} reviews, saved to pocketfm_reviews_cleaned.csv")

## Step 8 — Analysis: Rating Distribution

Where does the app actually stand with users, in aggregate?

In [ ]:
plt.figure()
sns.countplot(data=df, x="rating", palette="viridis")
plt.title("Distribution of Star Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.show()

print(df["rating"].value_counts(normalize=True).sort_index().mul(100).round(1))

## Step 9 — Rating Trend Over Time

This is the growth-relevant view: is user sentiment improving or declining month over month? A sustained dip is a retention/churn red flag worth investigating.

In [ ]:
monthly_avg = df.groupby("year_month")["rating"].mean().reset_index()
monthly_avg = monthly_avg.sort_values("year_month")

plt.figure()
sns.lineplot(data=monthly_avg, x="year_month", y="rating", marker="o")
plt.xticks(rotation=45)
plt.title("Average Rating by Month")
plt.xlabel("Month")
plt.ylabel("Average Rating")
plt.tight_layout()
plt.show()

## Step 10 — Sentiment Analysis (VADER)

We use VADER (tuned for short, informal text like reviews/social media) to score each English review, then compare sentiment against star rating — this cross-check is important: sometimes text sentiment and star rating disagree, which itself is an insight (e.g., a 5-star review that's actually complaining, or sarcastic).

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text, lang):
    if lang != "en":
        return np.nan
    score = analyzer.polarity_scores(text)["compound"]
    return score

df["sentiment_score"] = df.apply(lambda row: get_sentiment(row["review_text_clean"], row["language"]), axis=1)

def sentiment_label(score):
    if pd.isna(score):
        return "not scored (non-English)"
    elif score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment_score"].apply(sentiment_label)
df["sentiment_label"].value_counts()

In [ ]:
# Cross-check: does text sentiment agree with star rating?
en_df = df[df["language"] == "en"]

crosstab = pd.crosstab(en_df["rating_bucket"], en_df["sentiment_label"], normalize="index").mul(100).round(1)
print(crosstab)

crosstab.plot(kind="bar", stacked=True, figsize=(9,5), colormap="coolwarm")
plt.title("Sentiment Label vs Rating Bucket (English reviews)")
plt.ylabel("% of reviews")
plt.xticks(rotation=0)
plt.legend(title="Sentiment", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## Step 11 — Theme / Keyword Extraction from Negative Reviews

This is the most "growth analyst"-flavored step: what are people actually complaining about? We look at word frequency in 1-2 star English reviews, plus check for specific business-relevant keywords (pricing, ads, buffering, content repetition, coins) that map directly to product/growth levers.

In [ ]:
STOPWORDS = set("the a an is it this that and to of for in on with was were i my me app very not but so has have had you your are be as at from or if just can will would could its it's im i'm".split())

negative_en = en_df[en_df["rating_bucket"] == "Negative (1-2)"]

words = " ".join(negative_en["review_text_clean"]).split()
words = [w for w in words if w not in STOPWORDS and len(w) > 2]

word_freq = Counter(words).most_common(25)
pd.DataFrame(word_freq, columns=["word", "count"])

In [ ]:
wc_text = " ".join(negative_en["review_text_clean"])
wordcloud = WordCloud(width=900, height=450, background_color="white",
                       stopwords=STOPWORDS, colormap="Reds").generate(wc_text)

plt.figure(figsize=(11, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Most Common Words in Negative (1-2 star) Reviews")
plt.show()

In [ ]:
# Business-relevant keyword tagging — map raw complaints to specific product/growth levers
keyword_map = {
    "pricing_complaint": ["price", "expensive", "coin", "coins", "money", "pay", "paid", "costly", "subscription"],
    "buffering_bug": ["buffer", "buffering", "lag", "crash", "loading", "slow", "bug", "error"],
    "content_repetition": ["repeat", "repetitive", "same", "boring", "short", "old", "again"],
    "ads_complaint": ["ad", "ads", "advertisement", "advertisements"],
}

def tag_keywords(text):
    tags = []
    for label, kws in keyword_map.items():
        if any(kw in text for kw in kws):
            tags.append(label)
    return tags

en_df = en_df.copy()
en_df["complaint_tags"] = en_df["review_text_clean"].apply(tag_keywords)

# % of NEGATIVE reviews mentioning each theme
neg_tagged = en_df[en_df["rating_bucket"] == "Negative (1-2)"]
tag_counts = Counter([tag for tags in neg_tagged["complaint_tags"] for tag in tags])
tag_pct = {k: round(v / len(neg_tagged) * 100, 1) for k, v in tag_counts.items()}

pd.Series(tag_pct).sort_values(ascending=False).plot(kind="barh", color="crimson")
plt.title("% of Negative Reviews Mentioning Each Complaint Theme")
plt.xlabel("% of negative reviews")
plt.tight_layout()
plt.show()

print(tag_pct)

## Step 12 — Growth Insight Synthesis

Turn the analysis above into the kind of 3-4 line recommendation memo a Growth Analyst would actually send to a PM. **Fill in the blanks below with YOUR actual numbers once you've run the cells above** — don't leave the placeholders in when you show this in an interview.

**Template:**

> Across `[N]` reviews analyzed, average rating is `[X]`/5, with `[Y]`% of 1-2 star reviews. The most common complaint theme is **`[e.g., pricing_complaint]`**, appearing in `[Z]`% of negative reviews. This is a plausible churn driver, since users who feel forced to pay per-episode may disengage rather than convert to habitual paying users. Monthly rating trend shows `[improving/declining/stable]` sentiment over the last `[N]` months, coinciding with `[any app update/feature you noticed in the data]`.
>
> **Recommendation:** Run an A/B test on `[e.g., a bundled/flat pricing option for a subset of users]`, measuring impact on `[e.g., Day-7 retention, and revenue per user]`, before rolling out broadly.

This paragraph — with your real numbers dropped in — is exactly what you say when asked "walk me through a project" in the interview. It shows the full loop: data → insight → business action → how you'd validate it.

In [ ]:
# Quick summary stats to fill into the template above
print("Total reviews analyzed:", len(df))
print("Average rating:", round(df['rating'].mean(), 2))
print("% negative (1-2 star):", round((df['rating_bucket'] == 'Negative (1-2)').mean() * 100, 1))
print("Language mix:")
print(df['language'].value_counts(normalize=True).mul(100).round(1).head(5))
print("\nTop complaint theme in negative reviews:")
print(pd.Series(tag_pct).sort_values(ascending=False).head(1))

## You're done — what you now have

- A real, fetched-and-cleaned dataset (`pocketfm_reviews_cleaned.csv`) you can reference and extend
- A repeatable pipeline: fetch → clean (dedupe, dates, emojis, multilingual, text normalization) → feature engineer → analyze → sentiment → theme extraction → business recommendation
- A genuine, specific talking point about Pocket FM's actual users for your interview — not a generic Kaggle-dataset project

**Optional next steps if you have spare time:**
- Compare Pocket FM's rating trend against a competitor (e.g., Kuku FM, Audible) using the same pipeline
- Try topic modeling (LDA) instead of manual keyword tagging for a more advanced technique to mention
- Build a small Power BI/Excel dashboard on top of `pocketfm_reviews_cleaned.csv` — ties directly back to your existing Power BI project experience